# GCE NPI Benchmark Playbook

This runnable playbook guides you through building benchmark images and remotely orchestrating NPI benchmarks on a Google Compute Engine (GCE) VM. 

Unlike GKE, which runs Jobs natively, GCE requires us to execute `npi.py` which runs Docker containers on the target machine. This notebook executes those commands remotely via `gcloud compute ssh`.

### Instructions
1. Ensure your GCE VM is already created and running.
2. Run each cell sequentially.
3. If a cell fails, stop and resolve the error before continuing.

In [ ]:
# --- CONFIGURATION VARIABLES ---
# Replace these with your actual environment details before running any other cells.

PROJECT_ID = "YOUR_PROJECT_ID"
GCSFUSE_VERSION = "v3.5.6"  # Example: v3.5.6
BUCKET_NAME = "YOUR_BUCKET_NAME"
BQ_DATASET_ID = "YOUR_BQ_DATASET_ID"

# GCE Specific Variables
VM_NAME = "YOUR_VM_NAME"
VM_ZONE = "YOUR_VM_ZONE" # e.g., us-central1-a

## Step 1: Build Benchmark Images
This will use Google Cloud Build to construct the Docker images required for testing and upload them to Artifact Registry.

In [ ]:
!gcloud config set project {PROJECT_ID}

# Enable Artifact Registry API and create the repository (errors ignored if it already exists)
!gcloud services enable artifactregistry.googleapis.com --project={PROJECT_ID}
!gcloud artifacts repositories create gcsfuse-benchmarks --repository-format=docker --location=us --project={PROJECT_ID} || echo "Repository might already exist."

# Build the images using the Makefile
!make build PROJECT={PROJECT_ID} GCSFUSE_VERSION={GCSFUSE_VERSION}

# Verify Images
!gcloud artifacts docker images list us-docker.pkg.dev/{PROJECT_ID}/gcsfuse-benchmarks

## Step 2: Configure the Target GCE VM
Ensure the remote VM has the necessary tools (Docker, python3, lscpu) and copy the benchmarking tools over.

In [ ]:
# Install system dependencies
!gcloud compute ssh {VM_NAME} --zone={VM_ZONE} --project={PROJECT_ID} --command="sudo apt-get update && sudo apt-get install -y util-linux python3 docker.io"

# Ensure the user has permissions to run docker without sudo
!gcloud compute ssh {VM_NAME} --zone={VM_ZONE} --project={PROJECT_ID} --command="sudo usermod -aG docker \$USER"

# Authenticate docker to artifact registry so the VM can pull the benchmark images
!gcloud compute ssh {VM_NAME} --zone={VM_ZONE} --project={PROJECT_ID} --command="gcloud auth configure-docker us-docker.pkg.dev --quiet"

# Copy the benchmarking code to the VM
!gcloud compute scp --recurse ../npi ../fio {VM_NAME}:~ --zone={VM_ZONE} --project={PROJECT_ID}

## Step 3: Run the Benchmarks
Execute the `npi.py` script remotely on the VM. This script automatically downloads the images and orchestrates the benchmark test matrix sequentially.

In [ ]:
# We use 'sg docker -c' to ensure the user's new docker group membership is picked up without requiring them to log out and back in
benchmark_cmd = f"cd npi && sg docker -c 'python3 npi.py --benchmarks all --bucket-name {BUCKET_NAME} --project-id {PROJECT_ID} --bq-dataset-id {BQ_DATASET_ID} --gcsfuse-version {GCSFUSE_VERSION}'"

print(f"Executing: {benchmark_cmd} on {VM_NAME}...")

!gcloud compute ssh {VM_NAME} --zone={VM_ZONE} --project={PROJECT_ID} --command="{benchmark_cmd}"